In [ ]:
# cell 1
!pip install scikit-learn tensorflow pandas numpy matplotlib joblib

In [ ]:
# cell 2
from pathlib import Path
import json
import random
import shutil

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler

import tensorflow as tf
from tensorflow.keras import layers, models, callbacks

In [ ]:
# cell 3
# mount drive
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# Cell 4 — Reproducibility setup
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices("GPU"))

In [ ]:
# cell 5 - ddefine paths
PREPARED_DATA_DIR = Path("/content/drive/MyDrive/datasets/demo5/prepared_audio_data")

OUTPUT_DIR = Path("/content/drive/MyDrive/datasets/demo5/trained_audio_model")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FEATURES_PATH = PREPARED_DATA_DIR / "audio_features.npy"
LABELS_PATH = PREPARED_DATA_DIR / "audio_labels.npy"
METADATA_PATH = PREPARED_DATA_DIR / "audio_window_metadata.csv"
FEATURE_INFO_PATH = PREPARED_DATA_DIR / "feature_info.json"

print("Features exists:", FEATURES_PATH.exists())
print("Labels exists:", LABELS_PATH.exists())
print("Metadata exists:", METADATA_PATH.exists())
print("Feature info exists:", FEATURE_INFO_PATH.exists())
print("Output dir:", OUTPUT_DIR)

In [ ]:
# Cell 6 — Load prepared audio data
X = np.load(FEATURES_PATH)
y = np.load(LABELS_PATH)
metadata_df = pd.read_csv(METADATA_PATH)

with open(FEATURE_INFO_PATH, "r") as file:
    feature_info = json.load(file)

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Metadata shape:", metadata_df.shape)
print("Feature info:")
feature_info

In [ ]:
# Cell 7 — Check class distribution
label_counts = pd.Series(y).value_counts().sort_index()

print("Label counts:")
print(label_counts)

print("\nMetadata label distribution:")
print(metadata_df["label_name"].value_counts())

In [ ]:
# Cell 8 — Group-based train/val/test split
groups = metadata_df["video_id"].values

# First split: train+val vs test
gss_test = GroupShuffleSplit(
    n_splits=1,
    test_size=0.15,
    random_state=SEED
)

train_val_idx, test_idx = next(
    gss_test.split(X, y, groups=groups)
)

X_train_val = X[train_val_idx]
y_train_val = y[train_val_idx]
metadata_train_val = metadata_df.iloc[train_val_idx].reset_index(drop=True)

X_test = X[test_idx]
y_test = y[test_idx]
metadata_test = metadata_df.iloc[test_idx].reset_index(drop=True)

# Second split: train vs validation
groups_train_val = metadata_train_val["video_id"].values

gss_val = GroupShuffleSplit(
    n_splits=1,
    test_size=0.1765,  # about 15% of total after 85% train_val
    random_state=SEED
)

train_idx, val_idx = next(
    gss_val.split(X_train_val, y_train_val, groups=groups_train_val)
)

X_train = X_train_val[train_idx]
y_train = y_train_val[train_idx]
metadata_train = metadata_train_val.iloc[train_idx].reset_index(drop=True)

X_val = X_train_val[val_idx]
y_val = y_train_val[val_idx]
metadata_val = metadata_train_val.iloc[val_idx].reset_index(drop=True)

print("Train:", X_train.shape, y_train.shape)
print("Val:", X_val.shape, y_val.shape)
print("Test:", X_test.shape, y_test.shape)

print("\nTrain labels:")
print(pd.Series(y_train).value_counts())

print("\nVal labels:")
print(pd.Series(y_val).value_counts())

print("\nTest labels:")
print(pd.Series(y_test).value_counts())

print("\nUnique train videos:", metadata_train["video_id"].nunique())
print("Unique val videos:", metadata_val["video_id"].nunique())
print("Unique test videos:", metadata_test["video_id"].nunique())

In [ ]:
# Cell 9 — Save split metadata
metadata_train.to_csv(OUTPUT_DIR / "audio_train_window_metadata.csv", index=False)
metadata_val.to_csv(OUTPUT_DIR / "audio_val_window_metadata.csv", index=False)
metadata_test.to_csv(OUTPUT_DIR / "audio_test_window_metadata.csv", index=False)

np.save(OUTPUT_DIR / "X_audio_train.npy", X_train)
np.save(OUTPUT_DIR / "X_audio_val.npy", X_val)
np.save(OUTPUT_DIR / "X_audio_test.npy", X_test)

np.save(OUTPUT_DIR / "y_audio_train.npy", y_train)
np.save(OUTPUT_DIR / "y_audio_val.npy", y_val)
np.save(OUTPUT_DIR / "y_audio_test.npy", y_test)

print("Saved split files to:", OUTPUT_DIR)

In [ ]:
# Cell 10 — Create summary features for baseline model
def summarize_audio_sequence(X_sequence):
    """
    Converts sequence data:
    (samples, time_steps, feature_count)

    into static summary data:
    (samples, feature_count * 4)
    """

    mean_features = np.mean(X_sequence, axis=1)
    std_features = np.std(X_sequence, axis=1)
    min_features = np.min(X_sequence, axis=1)
    max_features = np.max(X_sequence, axis=1)

    return np.concatenate(
        [mean_features, std_features, min_features, max_features],
        axis=1
    )


X_train_summary = summarize_audio_sequence(X_train)
X_val_summary = summarize_audio_sequence(X_val)
X_test_summary = summarize_audio_sequence(X_test)

print("X_train_summary:", X_train_summary.shape)
print("X_val_summary:", X_val_summary.shape)
print("X_test_summary:", X_test_summary.shape)

In [ ]:
# Cell 11 — Train Random Forest baseline
rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_split=4,
    min_samples_leaf=2,
    class_weight="balanced",
    random_state=SEED,
    n_jobs=-1
)

rf_model.fit(X_train_summary, y_train)

val_probs_rf = rf_model.predict_proba(X_val_summary)[:, 1]
val_preds_rf = (val_probs_rf >= 0.5).astype(int)

test_probs_rf = rf_model.predict_proba(X_test_summary)[:, 1]
test_preds_rf = (test_probs_rf >= 0.5).astype(int)

print("Validation classification report:")
print(classification_report(y_val, val_preds_rf, target_names=["truthful", "deceptive"]))

print("Test classification report:")
print(classification_report(y_test, test_preds_rf, target_names=["truthful", "deceptive"]))

In [ ]:
# Cell 12 — Baseline metrics function
def calculate_metrics(y_true, y_pred, y_prob):
    metrics = {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
    }

    try:
        metrics["roc_auc"] = float(roc_auc_score(y_true, y_prob))
    except Exception:
        metrics["roc_auc"] = None

    return metrics


rf_val_metrics = calculate_metrics(y_val, val_preds_rf, val_probs_rf)
rf_test_metrics = calculate_metrics(y_test, test_preds_rf, test_probs_rf)

print("RF validation metrics:")
print(rf_val_metrics)

print("\nRF test metrics:")
print(rf_test_metrics)

In [ ]:
# Cell 13 — Save baseline model
joblib.dump(rf_model, OUTPUT_DIR / "audio_random_forest_baseline.pkl")

baseline_metrics = {
    "model": "RandomForestClassifier",
    "validation": rf_val_metrics,
    "test": rf_test_metrics,
    "input_type": "window_summary_features",
    "summary_features": ["mean", "std", "min", "max"]
}

with open(OUTPUT_DIR / "audio_baseline_metrics.json", "w") as file:
    json.dump(baseline_metrics, file, indent=4)

print("Saved Random Forest baseline.")

In [ ]:
# Cell 14 — Build BiLSTM model
time_steps = X_train.shape[1]
feature_count = X_train.shape[2]

print("Time steps:", time_steps)
print("Feature count:", feature_count)

def build_audio_bilstm_model(time_steps, feature_count):
    inputs = layers.Input(shape=(time_steps, feature_count))

    x = layers.Masking(mask_value=0.0)(inputs)

    x = layers.Bidirectional(
        layers.LSTM(
            64,
            return_sequences=True,
            dropout=0.25,
            recurrent_dropout=0.0
        )
    )(x)

    x = layers.Bidirectional(
        layers.LSTM(
            32,
            return_sequences=False,
            dropout=0.25,
            recurrent_dropout=0.0
        )
    )(x)

    x = layers.Dense(64, activation="relu")(x)
    x = layers.Dropout(0.35)(x)

    x = layers.Dense(32, activation="relu")(x)
    x = layers.Dropout(0.25)(x)

    outputs = layers.Dense(1, activation="sigmoid")(x)

    model = models.Model(inputs=inputs, outputs=outputs)

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
        loss="binary_crossentropy",
        metrics=[
            "accuracy",
            tf.keras.metrics.AUC(name="auc"),
            tf.keras.metrics.Precision(name="precision"),
            tf.keras.metrics.Recall(name="recall"),
        ]
    )

    return model


audio_bilstm_model = build_audio_bilstm_model(time_steps, feature_count)
audio_bilstm_model.summary()

In [ ]:
# Cell 15 — Compute class weights
unique, counts = np.unique(y_train, return_counts=True)
class_counts = dict(zip(unique, counts))

total = len(y_train)
class_weight = {
    0: total / (2 * class_counts.get(0, 1)),
    1: total / (2 * class_counts.get(1, 1)),
}

print("Class counts:", class_counts)
print("Class weights:", class_weight)

In [ ]:
# Cell 16 — Train BiLSTM
checkpoint_path = OUTPUT_DIR / "best_audio_bilstm_model.keras"

training_callbacks = [
    callbacks.ModelCheckpoint(
        filepath=str(checkpoint_path),
        monitor="val_auc",
        mode="max",
        save_best_only=True,
        verbose=1
    ),
    callbacks.EarlyStopping(
        monitor="val_auc",
        mode="max",
        patience=12,
        restore_best_weights=True,
        verbose=1
    ),
    callbacks.ReduceLROnPlateau(
        monitor="val_auc",
        mode="max",
        factor=0.5,
        patience=5,
        min_lr=1e-6,
        verbose=1
    )
]

history = audio_bilstm_model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=80,
    batch_size=16,
    class_weight=class_weight,
    callbacks=training_callbacks,
    verbose=1
)

In [ ]:
# Cell 17 — Plot training curves
history_df = pd.DataFrame(history.history)
history_df.to_csv(OUTPUT_DIR / "audio_bilstm_training_history.csv", index=False)

plt.figure(figsize=(8, 5))
plt.plot(history_df["loss"], label="train_loss")
plt.plot(history_df["val_loss"], label="val_loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Audio BiLSTM Loss")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(history_df["auc"], label="train_auc")
plt.plot(history_df["val_auc"], label="val_auc")
plt.xlabel("Epoch")
plt.ylabel("AUC")
plt.title("Audio BiLSTM AUC")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Cell 18 — Evaluate BiLSTM
best_audio_model = tf.keras.models.load_model(OUTPUT_DIR / "best_audio_bilstm_model.keras")

val_probs_bilstm = best_audio_model.predict(X_val).ravel()
val_preds_bilstm = (val_probs_bilstm >= 0.5).astype(int)

test_probs_bilstm = best_audio_model.predict(X_test).ravel()
test_preds_bilstm = (test_probs_bilstm >= 0.5).astype(int)

print("Validation classification report:")
print(classification_report(y_val, val_preds_bilstm, target_names=["truthful", "deceptive"]))

print("Test classification report:")
print(classification_report(y_test, test_preds_bilstm, target_names=["truthful", "deceptive"]))

bilstm_val_metrics = calculate_metrics(y_val, val_preds_bilstm, val_probs_bilstm)
bilstm_test_metrics = calculate_metrics(y_test, test_preds_bilstm, test_probs_bilstm)

print("BiLSTM validation metrics:")
print(bilstm_val_metrics)

print("\nBiLSTM test metrics:")
print(bilstm_test_metrics)

In [ ]:
# Cell 19 — Confusion matrices
print("Random Forest test confusion matrix:")
print(confusion_matrix(y_test, test_preds_rf))

print("\nBiLSTM test confusion matrix:")
print(confusion_matrix(y_test, test_preds_bilstm))

In [ ]:
# Cell 20 — Save final BiLSTM model and metrics
final_model_path = OUTPUT_DIR / "final_audio_bilstm_model.keras"
best_audio_model.save(final_model_path)

audio_bilstm_metrics = {
    "model": "Audio_BiLSTM",
    "input_shape": {
        "time_steps": int(time_steps),
        "feature_count": int(feature_count)
    },
    "validation": bilstm_val_metrics,
    "test": bilstm_test_metrics,
    "training_config": {
        "epochs": 80,
        "batch_size": 16,
        "learning_rate": 1e-4,
        "class_weight": {
            str(k): float(v) for k, v in class_weight.items()
        }
    }
}

with open(OUTPUT_DIR / "audio_bilstm_metrics.json", "w") as file:
    json.dump(audio_bilstm_metrics, file, indent=4)

print("Saved final model:", final_model_path)
print("Saved metrics:", OUTPUT_DIR / "audio_bilstm_metrics.json")

In [ ]:
# Cell 21 — Compare baseline vs BiLSTM
comparison_df = pd.DataFrame([
    {
        "model": "Random Forest Baseline",
        **rf_test_metrics
    },
    {
        "model": "Audio BiLSTM",
        **bilstm_test_metrics
    }
])

comparison_df.to_csv(OUTPUT_DIR / "audio_model_comparison.csv", index=False)

comparison_df

In [ ]:
# Cell 22 — Save model package info
model_package_info = {
    "prepared_data_source": str(PREPARED_DATA_DIR),
    "output_dir": str(OUTPUT_DIR),
    "models_saved": {
        "random_forest_baseline": "audio_random_forest_baseline.pkl",
        "best_audio_bilstm": "best_audio_bilstm_model.keras",
        "final_audio_bilstm": "final_audio_bilstm_model.keras"
    },
    "metrics_saved": {
        "baseline_metrics": "audio_baseline_metrics.json",
        "bilstm_metrics": "audio_bilstm_metrics.json",
        "comparison": "audio_model_comparison.csv",
        "training_history": "audio_bilstm_training_history.csv"
    },
    "feature_info": feature_info
}

with open(OUTPUT_DIR / "audio_model_package_info.json", "w") as file:
    json.dump(model_package_info, file, indent=4)

print("Audio model package complete.")
for path in OUTPUT_DIR.iterdir():
    print(path.name)